# Efektywne programowanie w języku Python

## Efektywnie 2

### Od part02 do poprawnego i mierzalnego rozwiązania

`part02` wyjaśnia kolekcje, kopiowanie, iterację i comprehensions. Tutaj sprawdzamy skutki uboczne oraz koszt alternatywnych rozwiązań.

Procedura z `opt01`: **wymagania → zgodność wyników i skutków ubocznych → pomiar → interpretacja**.

Główne tematy: referencje, wartości domyślne, listy zagnieżdżone, zbiory i filtrowanie. Indeksowanie przez `~` oraz nanosekundowe mikrobenchmarki są opcjonalne.

## 1. Referencje: mutacja a przypisanie

Nazwy wskazują obiekty. `b = a` nie kopiuje listy. Przy wywołaniu funkcji parametry zostają związane z przekazanymi obiektami — często nazywa się to *call by sharing*.

Reguła jest wspólna dla list, liczb i napisów. Różnica dotyczy mutowalności obiektów, a nie osobnego sposobu przekazywania `int`, `float` i `str`.

In [ ]:
a = [1, 2, 3]
b = a
b.append(10)
print(a, b)
assert b is a and a == [1, 2, 3, 10]

### Parametr funkcji też jest nazwą

Mutacja przekazanej listy może być widoczna u wywołującego. Ponowne przypisanie parametru zmienia lokalne powiązanie, nie powiązanie nazwy wywołującego.

In [ ]:
def mutate(values):
    values.append(99)

def rebind(values):
    values = [0]  # Nowe lokalne powiązanie.
    return values

items = [1, 2]
mutate(items)
replacement = rebind(items)
print(items, replacement)
assert items == [1, 2, 99] and replacement == [0]

### Niemutowalna liczba

`number += 1` wiąże parametr z wynikiem dodawania. Nie zmienia istniejącego obiektu `int`. Dla listy `+=` może natomiast modyfikować obiekt — znaczenie zależy od typu.

In [ ]:
def increment(number):
    number += 1
    return number

value = 10
assert increment(value) == 11
assert value == 10
print(value)

## 2. Pusta kolekcja: znaczenie warunku

Dla zwykłej listy `if data:` oznacza, że ma elementy; `if not data:` — że jest pusta. Sprawdzamy oba przebiegi zamiast testować tylko pustą listę.

In [ ]:
def process(values):
    return sum(values)

for data in ([], [0], [1, 2, 3]):
    if data:
        print(f"{data}: wynik {process(data)}")
    else:
        print("Brak danych")
assert process([1, 2, 3]) == 6

## 3. Zbiór a sprawdzanie znaków

Wymaganie: napis zawiera wyłącznie znaki z `BCDGIJLMNOPSUVWZ`. Wielkość liter ma znaczenie. Pusty napis uznajemy za zgodny: nie zawiera niedozwolonego znaku.

Trzy warianty mają dawać ten sam `bool`. Nie normalizujemy danych i nie zmieniamy tego wymagania w trakcie optymalizacji.

In [ ]:
ALLOWED_TEXT = "BCDGIJLMNOPSUVWZ"
ALLOWED_SET = frozenset(ALLOWED_TEXT)

def allowed_loop(word):
    for letter in word:
        if letter not in ALLOWED_TEXT:
            return False
    return True

In [ ]:
def allowed_subset(word):
    return set(word) <= ALLOWED_SET

def allowed_all(word):
    return all(letter in ALLOWED_SET for letter in word)

letter_methods = {"pętla + napis": allowed_loop,
                  "set(word)": allowed_subset, "all + zbiór": allowed_all}

### Sprawdź kontrakt przed pomiarem

Powtórzenia liter nie zmieniają wyniku. `all()` i pętla mogą zakończyć pracę na pierwszym niedozwolonym znaku. `set(word)` najpierw przetwarza całe wejście.

Jeśli wymagamy niepustego słowa, dopisz `bool(word) and ...` we wszystkich wariantach i zmień odpowiedni test.

In [ ]:
letter_cases = {"": True, "BCD": True, "BBCD": True,
                "ABC": False, "bcd": False, "123": False}
for word, expected in letter_cases.items():
    assert all(method(word) == expected for method in letter_methods.values())
    print(f"{word!r}: {expected}")

### Kiedy zbiór może się opłacić?

Tutaj napis dozwolonych znaków ma stałą, małą długość. Koszt `set(word)` i miejsce pierwszego błędu mogą decydować o wyniku.

Przygotowanie `ALLOWED_SET` wykonaliśmy raz. W pomiarze zapytań nie doliczamy go ponownie; osobny eksperyment pokazuje koszt przygotowania przy pojedynczym sprawdzeniu.

In [ ]:
import sys
import timeit
from statistics import median

print(sys.version)
print(sys.executable)

def measure_action(action):
    timer = timeit.Timer(action)
    loops, _ = timer.autorange()
    samples = [elapsed / loops for elapsed in timer.repeat(3, loops)]
    return median(samples), min(samples), max(samples)

### Przygotowane narzędzie pomiarowe

Funkcja `measure_action` przyjmuje operację bez argumentów. `lambda` poniżej przygotowuje taką operację dla wybranych danych.

Kalibrujemy liczbę wykonań i powtarzamy pomiar trzy razy. Pokazujemy medianę oraz zakres; wynik odnosi się do wersji Pythona i komputera podanych wyżej.

In [ ]:
valid_word = "BCDGI" * 1_000
timing_words = {"krótkie": "BCD", "długie zgodne": valid_word,
                "błąd na początku": "A" + valid_word,
                "błąd na końcu": valid_word + "A"}
letter_results = []
for case, word in timing_words.items():
    for label, method in letter_methods.items():
        sample = measure_action(lambda method=method, word=word: method(word))
        letter_results.append((case, label, *sample))

In [ ]:
for case, label, typical, fastest, slowest in letter_results:
    print(f"{case:>16}, {label:<14}: {typical * 1e6:>8.2f} µs "
          f"[{fastest * 1e6:.2f}, {slowest * 1e6:.2f}]")

### Koszt przygotowania zbioru

Poniżej porównujemy pojedyncze sprawdzenie z gotowym zbiorem i z budowaniem zbioru dozwolonych znaków za każdym razem. Wynik logiczny jest ten sam; zakres mierzonej pracy różni się przygotowaniem danych.

In [ ]:
short_word = "BCD"
assert set(short_word) <= ALLOWED_SET
prepared = measure_action(lambda: set(short_word) <= ALLOWED_SET)
with_setup = measure_action(lambda: set(short_word) <= frozenset(ALLOWED_TEXT))
print(f"Gotowy zbiór: {prepared[0] * 1e6:.3f} µs")
print(f"Z przygotowaniem: {with_setup[0] * 1e6:.3f} µs")

## 4. Wartości domyślne: wspólna lista

Wartości domyślne są obliczane podczas wykonania `def`, nie przy każdym wywołaniu. Mutowalny obiekt może być współdzielony między wywołaniami.

Poniższy kod celowo pokazuje pułapkę. Jeśli każde wywołanie ma zaczynać z nową listą, wspólna wartość domyślna nie spełnia wymagania.

In [ ]:
def append_bad(values=[]):
    values.append(1)
    return values

first_bad = append_bad()
print(first_bad.copy())  # Migawka: [1].
second_bad = append_bad()
print(second_bad.copy())  # Migawka: [1, 1].
assert first_bad is second_bad
assert first_bad == [1, 1]  # Pierwszy wynik też się zmienił.

### Nowa lista przy pominiętym argumencie

`None` jest znacznikiem pominiętej listy. Tworzymy nową listę tylko wtedy, gdy argument jest `None`.

Nie używaj `if not values:` do tego celu — jawnie przekazana pusta lista też jest falsy. Ustalamy, że przekazana lista ma zostać zmodyfikowana.

In [ ]:
def append_good(values=None):
    if values is None:
        values = []
    values.append(1)
    return values

first_good = append_good()
second_good = append_good()
assert first_good == second_good == [1]
assert first_good is not second_good
print(first_good, second_good)

In [ ]:
provided = []
returned = append_good(provided)
assert returned is provided and provided == [1]
print(provided, returned)
# Jeśli funkcja ma zachować wejście, potrzebujemy innego kontraktu i kopii.

## 5. Lista list: powielasz referencje

`[[]] * 5` tworzy listę pięciu odwołań do jednego wewnętrznego obiektu. Powtórka z `part02` służy tu sprawdzeniu skutków ubocznych.

In [ ]:
shared_rows = [[]] * 5
shared_rows[0].append(1)
print(shared_rows)
assert shared_rows[0] is shared_rows[1]
assert shared_rows == [[1], [1], [1], [1], [1]]

### Osobne listy dla osobnych wierszy

In [ ]:
independent_rows = [[] for _ in range(5)]
independent_rows[0].append(1)
print(independent_rows)
assert independent_rows[0] is not independent_rows[1]
assert independent_rows == [[1], [], [], [], []]

## 6. Równe klucze słownika

`0 == False` oraz `1 == True`. Każda z tych par ma też taki sam hash. W słowniku równe klucze wskazują ten sam wpis; późniejsze przypisanie zastępuje jego wartość.

Podobnie `1`, `1.0` i `True` nie tworzą trzech niezależnych wpisów. Nie mieszaj tych typów kluczy, jeśli ich rozróżnienie jest wymaganiem.

In [ ]:
numbers_first = {0: "pierwsza", False: "ostatnia", 1: "foo", True: "bar"}
bools_first = {False: "pierwsza", 0: "ostatnia", True: "foo", 1: "bar"}
print(numbers_first)
print(bools_first)
assert len(numbers_first) == len(bools_first) == 2
assert numbers_first == bools_first
assert hash(0) == hash(False) and hash(1) == hash(True)

### Reprezentacja klucza a wartość wpisu

W tym pokazie CPython zachowuje pierwotny obiekt klucza, a kolejne równe klucze zastępują wartość. Dlatego wydruk może pokazywać `0` albo `False`.

Nie jest to kolizja hashy dwóch różnych wartości: klucze są także równe. Sama równość hashy nie wystarcza, aby utożsamić wpisy.

In [ ]:
equivalent = {1: "int", 1.0: "float", True: "bool"}
assert len(equivalent) == 1
assert equivalent[1] == equivalent[1.0] == equivalent[True] == "bool"
print(equivalent)

## 7. Filtrowanie: te same dane, ten sam wynik

Wymaganie: zwrócić listę nieparzystych liczb w kolejności wejściowej. Porównujemy pętlę, `filter()` z funkcją `lambda` i listę składaną.

Każdy wariant materializuje wynik jako listę. Samo utworzenie leniwego `filter()` nie wykonałoby tej samej pracy.

In [ ]:
def odds_loop(values):
    result = []
    for value in values:
        if value % 2:
            result.append(value)
    return result

def odds_filter(values):
    return list(filter(lambda value: value % 2, values))

In [ ]:
def odds_comprehension(values):
    return [value for value in values if value % 2]

filter_methods = {"pętla": odds_loop, "filter + lambda": odds_filter,
                  "lista składana": odds_comprehension}
for values, expected in (([], []), ([0, 2], []), ([1], [1]),
                         ([-3, -2, -1, 0, 1], [-3, -1, 1])):
    assert all(method(values) == expected for method in filter_methods.values())
print("Warianty zgodne dla pustego wejścia, znaków liczb i parzystości")

### Mierz kilka rozmiarów

Przygotowanie listy wejściowej jest poza pomiarem; tworzenie listy wynikowej jest wewnątrz. Dane wejściowe pozostają niezmienione.

Sprawdzenie dużych wyników robimy przed pomiarem. Nie traktuj zapisanego wyniku sprzed lat jako aktualnego rankingu.

In [ ]:
filter_results = []
for size in (10, 10_000, 1_000_000):
    values = list(range(size))
    expected = list(range(1, size, 2))
    for label, method in filter_methods.items():
        assert method(values) == expected
        sample = measure_action(lambda method=method, values=values: method(values))
        filter_results.append((size, label, *sample))
    assert values == list(range(size))

In [ ]:
for size, label, typical, fastest, slowest in filter_results:
    print(f"n={size:>7}, {label:<16}: {typical * 1e6:>10.2f} µs "
          f"[{fastest * 1e6:.2f}, {slowest * 1e6:.2f}]")

### Zinterpretuj pomiar

- Który wariant wybrałbyś dla małych danych, a który dla miliona elementów?
- Czy różnica czasu uzasadnia mniej czytelny zapis?
- Co wchodzi do pomiaru: wywołania funkcji, iteracja, tworzenie wyniku?

CPython 3.12 wprowadził inlining list, dict i set comprehensions (PEP 709). Ranking zależy od wersji; przykłady składni są zgodne z Pythonem 3.10+.

## 8. Ciekawostka opcjonalna: operator ~

Dla liczby całkowitej `i` operator bitowej negacji spełnia `~i == -i - 1`. Może wskazać element liczony od końca, ale `a[-1]` lub `a[-2]` zwykle czyta się łatwiej.

Nie używaj `~` zamiast logicznego `not`. `~bool` jest wycofywane od Pythona 3.12 i ma zgłaszać błąd od 3.16.

In [ ]:
letters = ["a", "b", "c", "d", "e"]
print(letters[0], letters[1], letters[-1], letters[-2])
for index in range(len(letters)):
    assert ~index == -index - 1
    assert letters[~index] == letters[-index - 1]
print(letters[~0], letters[~1])

### Celowy błąd: indeks poza zakresem

In [ ]:
# Lista ma pięć elementów; najdalszy ujemny indeks to -5.
letters[-6]

## 9. Opcjonalny mikrobenchmark: trzy różne pytania

`bool(x)` sprawdza truthiness; `x == True` — równość z `True`; `x is True` — tożsamość z singletonem `True`.

Przed pomiarem ustal, którego zachowania potrzebuje program. Dla rzeczywistego `bool` wyniki mogą być zgodne; dla innych obiektów już nie.

In [ ]:
for value in (True, False, 1, 2, "True", [], [0], None):
    print(f"{value!r:>6}: == True {value == True!s:5}, "
          f"is True {value is True!s:5}, bool {bool(value)}")
one = 1
assert one == True and one is not True
assert bool(2) and not (2 == True)

### Pomiar tylko dla x = True

W tym ograniczonym przypadku warunki prowadzą do tej samej gałęzi. Mierzymy koszt samego sprawdzenia; wyniku nie przenosimy na inne typy ani na czas całej aplikacji.

Składnia `%%timeit -r3` powtarza pomiar trzy razy; liczba wykonań jest dobierana automatycznie.

In [ ]:
x = True

In [ ]:
%%timeit -r3
if x == True:
    pass

In [ ]:
%%timeit -r3
if x is True:
    pass

In [ ]:
%%timeit -r3
if x:
    pass

### Opcjonalnie: trzy sposoby sprawdzania pustej listy

Dla zwykłej listy te warunki są zgodne. Nie przenoś tej równoważności na dowolne obiekty z własnym porównywaniem czy wartością logiczną.

Domyślnym czytelnym zapisem pozostaje `if not values:`; nanosekundowe różnice nie przesądzają o wydajności całego programu.

In [ ]:
for values in ([], [0], [1, 2]):
    assert (len(values) == 0) == (values == []) == (not values)
alist = []

In [ ]:
%%timeit -r3
if len(alist) == 0:
    pass

In [ ]:
%%timeit -r3
if alist == []:
    pass

In [ ]:
%%timeit -r3
if not alist:
    pass

### Opcjonalnie: literały a konstruktory

`[]` i `list()` tworzą nowe puste listy; `{}` i `dict()` — nowe puste słowniki, przy standardowych powiązaniach nazw konstruktorów.

To pomiar kosztu utworzenia pustego obiektu. W praktycznym kodzie wybieramy prosty zapis; `list(iterable)` i `dict(pairs)` mają dodatkowe zastosowania, których literały puste nie zastępują.

In [ ]:
%%timeit -r3
[]

In [ ]:
%%timeit -r3
list()

In [ ]:
%%timeit -r3
{}

In [ ]:
%%timeit -r3
dict()

## Ćwiczenie końcowe: poprawność, czytelność czy czas?

Przyporządkuj zmiany do ich celu; niektóre dotyczą kilku celów:

1. Zastąpienie domyślnego `[]` znacznikiem `None`.
2. Zamiana `[[]] * n` na `[[] for _ in range(n)]`.
3. Przygotowanie zbioru dozwolonych liter raz dla wielu zapytań.
4. Zastąpienie pętli listą składaną po potwierdzeniu zgodności wyników.
5. Zamiana `x == True` na `if x` dla wejść `True`, `1` i `2`.

Dla ostatniego punktu najpierw sprawdź, czy zachowanie pozostaje zgodne z wymaganiem.

### Wnioski — po dyskusji

1–2: usuwają niezamierzone współdzielenie obiektów.

3: może zmniejszyć koszt wielu zapytań; przygotowanie też kosztuje.

4: może poprawić czytelność i czas; wynik sprawdzamy i mierzymy.

5: zmienia zachowanie dla `2`. O wyborze warunku decyduje znaczenie, nie ranking szybkości.

## Dokumentacja i źródła

- [Argumenty domyślne — tutorial Pythona](https://docs.python.org/3.10/tutorial/controlflow.html#default-argument-values).
- [Kolekcje, truthiness i klucze słownika](https://docs.python.org/3.10/library/stdtypes.html).
- [timeit — sposób pomiaru](https://docs.python.org/3.10/library/timeit.html).
- [Python 3.12 — PEP 709](https://docs.python.org/3.12/whatsnew/3.12.html#pep-709-comprehension-inlining).
- [bool — ograniczenia operatora ~](https://docs.python.org/3/builtins/stdtypes.html#boolean-type-bool).

Przykłady są zgodne z Pythonem 3.10+; informacja o `~bool` dotyczy nowszych wydań.

### Materiały historyczne

- [Sebastian Witowski — Writing faster Python](https://www.youtube.com/watch?v=YjHsOrOOSuI).
- [Florian Rohrer — What is the most confusing thing to you in Python?](https://dev.to/r0f1/what-is-the-most-confusing-thing-to-you-in-python).

Sprawdzone 2026-10-12: artykuł jest dostępny, a film odtwarza się. Starsze materiały są inspiracją do przykładów; wyniki czasów i szczegóły implementacyjne sprawdzaj w swoim środowisku i oficjalnej dokumentacji.